# Plot and evaluate summed/ extrapolated scores: Erosion

In [ ]:
## IMPORTS

%load_ext autoreload
%autoreload 2

import os
import sys

from torchvision import transforms
from torch.utils.data import DataLoader
import torch


import os
import numpy as np
from torch.utils.data import Dataset

import matplotlib.pyplot as plt
import random
from pathlib import Path 


# from ra_utils.autoscora.autoscorRA_Pipeline.scoring.src.io_scoring_method import io_scoring
# from ra_utils.autoscora.autoscorRA_Pipeline.scoring.src.run_utils import (
#     paths_list_scores_list_from_score_types,
#     restructure_paths_and_scores,
#     restructure_paths_and_scores_v2
# )

import pandas as pd
from typing import List 

import ra_utils
import ra_utils.utils.config_parser
import ra_utils.networks.loss_function


import ra_utils.data.dataloader_CR_patches
from ra_utils.data.dataloader_CR_patches import (
    load_img_SHS_patch_data,
    df_scores_to_dct_list,
)

import ra_utils.data.dataloader_CR_patches
from monai.data import Dataset, DataLoader
from monai.transforms import (
    Compose
)
import torch
from torch.utils.data import WeightedRandomSampler, DataLoader


from ra_utils.data.dataloader_CR_patches import (
    load_img_SHS_patch_data,
    dataset_and_loader,
    dataset_and_loader_several,
    df_scores_to_dct_list,
    make_paths_dataframe,
    restructure_paths_and_scores,
    restructure_paths_and_scores_v2,
    exclude_ROIS_according_surgery_status,
    split_training_val_test__on_patient_level,
    process_several_score_groups,
    process_single_score_group,
    load_img_SHS_patch_data
)

import yaml
from importlib import resources


import ra_utils.networks.architecture
from ra_utils.networks.architecture import (
    ResNet18Encoder,
    ResNet34Encoder,
    ResNet50Encoder,
    make_mlp,
    EncoderClassifierNetwork,
    MultiModalImageScoreTypeNetwork,
    ROI_type_encoder,
    model_interface_forward
)
import numpy as np
from ra_utils.autoscora.autoscorRA_Pipeline.scoring.src.network import Custom_VGG
from ra_utils.utils.utils_SHS_scoring import get_classes
import ra_utils.utils.utils
import torch.nn as nn


import ra_utils.mtan.im2im_pred.model_resnet_mtan.resnet_recon_mtan

from ra_utils.mtan.im2im_pred.model_resnet_mtan.resnet_recon_mtan import (
    #MTANResNetRecon
    MTANReconCls,
    build_mtan_recon_cls
)


from ra_utils.data.data_utils import (
    extract_extras_from_filename
)


from ra_utils.training.scores_SHS.model_builders import build_models_AE
from tqdm.notebook import tqdm

from ra_utils.progressionlearning.models.builder import (
    build_MTANAE
)
from ra_utils.progressionlearning.models.MTANUNet import (
    MTANRecUnet, 
    MTANRecUnet_v2,
    MTANRecUnet_v3
)
import monai
from monai.networks.nets import BasicUNet, UNet


from ra_utils.progressionlearning.models.builder import (
    build_MTANAE, 
    build_MTANAE_v2
)
from ra_utils.progressionlearning.models.MTANUNet import (
    MTANRecUnet,
    MTANRecUnet_v2
)
import monai
import monai.networks.nets
from monai.networks.nets import BasicUNet, UNet
from typing import Dict, List

import ra_utils.mtan.im2im_pred.model_resnet_mtan.resnet_mtan
from ra_utils.training.scores_SHS.model_builders import build_models_AE_v2, build_models_AE_v1_and2


# wrap model: 

from ra_utils.networks.architecture import (
    MultiModalImageScoreTypeNetworkAE,
    ROI_type_encoder
)



# get all scores types / roi types

import pingouin as pg


from ra_utils.data.shap_sums import (
    add_JSN_ERO_sums, 
    double_scoring_make_merge_id,
    limit_treatment_number
)


from ra_utils.data.icc import compute_icc3

from ra_utils.training.scores_SHS.scores_SHS_training_lib import (
    calculate_some_classification_metrics
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
)


import os
import mlflow
from mlflow.tracking import MlflowClient
import json
import yaml
from sklearn.metrics import balanced_accuracy_score

from pprint import pprint

# CODE: 
from ra_utils.evaluation.single_SHS import (
    combine_predictions, 
    get_main_metrics
)


from ra_utils.data.shap_sums import (
   #max_possible_score, 
    sum_and_extrapolate_scores_df,
    generate_score_differences,
    sum_and_extrapolate_scores_df_ERO_H,
    sum_and_extrapolate_scores_df_ERO_F,
    sum_and_extrapolate_scores_df_JSN_F,
    sum_and_extrapolate_scores_df_JSN_H
)



import ra_utils.visualization.plot_SHS_scores
from ra_utils.visualization.plot_SHS_scores import (
    plot_SHS_deltas, 
    plot_SHS_sums
)


with resources.files("ra_utils.resources.scores_metadata").joinpath("roi_scores_matching.csv") as f:
    df_scores_meta = pd.read_csv(f)
H_ERO_scores = sorted(df_scores_meta[(df_scores_meta["ERO_or_JSN"] == "ERO") & (df_scores_meta["region"] == "H")]["score_name"].unique())
F_ERO_scores = sorted(df_scores_meta[(df_scores_meta["ERO_or_JSN"] == "ERO") & (df_scores_meta["region"] == "F")]["score_name"].unique())
H_JSN_scores = sorted(df_scores_meta[(df_scores_meta["ERO_or_JSN"] == "JSN") & (df_scores_meta["region"] == "H")]["score_name"].unique())
F_JSN_scores = sorted(df_scores_meta[(df_scores_meta["ERO_or_JSN"] == "JSN") & (df_scores_meta["region"] == "F")]["score_name"].unique())



from sklearn.metrics import cohen_kappa_score
from rpy2.robjects import DataFrame, FloatVector, IntVector
from rpy2.robjects.packages import importr
r_icc = importr("ICC")
r_irr = importr("irr")
r_iccp = importr("psych")

In [ ]:
input_constants_yml_path = "/home/cwatzenboeck/code/RA/ra_utils/ra_utils/autoscora/autoscorRA_Pipeline/input/constants/input_constants_cw.yml"        
with open(input_constants_yml_path, "r") as f:
    const = yaml.safe_load(f)

In [ ]:
mlflow_runs_dir = "/home/cwatzenboeck/data/mlflow_cirpc_tmp/RA/data/"
os.environ["MLFLOW_TRACKING_URI"] = mlflow_runs_dir

client = MlflowClient()


In [ ]:
## Input: 

runs_to_examine = {
    "H_ERO_01": {
        #"Run ID": "",
        "predictions_raw__test": "/home/cwatzenboeck/data/mlflow_cirpc_tmp/RA/data/190700519300523579/d377515686f1423b8f8f197a32c3eaf5/artifacts/predictions/test_/tmpj88ccxjy.npz",
    },
    "F_ERO_01": {
        #"Run ID": "",
        "Notes": "RMTANAEv2_SP_MH_FL_LR2_pp",
        "predictions_raw__test": "/home/cwatzenboeck/data/mlflow_cirpc_tmp/RA/data/973476768866622660/438c7be819a74f27b429129088e92c37/artifacts/predictions/test_/scores.npz",
    },
    "JSN_01": {
        #"Run ID": "",
        "Notes": "Trained on JSN H +F",
        "predictions_raw__test": "/home/cwatzenboeck/data/mlflow_cirpc_tmp/RA/data/688231188141730244/30684b847c6b47abac078e7890abe84f/artifacts/predictions/test_/scores.npz",
    },

}


run = "H_ERO_01"
predictions_path = runs_to_examine[run]["predictions_raw__test"]
src = predictions_path

df = combine_predictions([src])
df["patientId_date_HF_LR"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:4]))
df["patientId_date_HF"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:3]))
df["patientId_date"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:2]))


# For now correct predictions... Class 6 can not occur anyhow...
df["labels"] = df["labels"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment="over_limit_to_NA"))
df["preds"] = df["preds"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment="over_limit_to_NA"))
df = df.dropna()
df_ERO_H  = df


#-------------------
run = "F_ERO_01"
predictions_path = runs_to_examine[run]["predictions_raw__test"]
src = predictions_path

df = combine_predictions([src])
df["patientId_date_HF_LR"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:4]))
df["patientId_date_HF"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:3]))
df["patientId_date"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:2]))


# For now correct predictions... Class 6 can not occur anyhow...
df["labels"] = df["labels"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment="over_limit_to_NA"))
df["preds"] = df["preds"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment="over_limit_to_NA"))
df = df.dropna()
df_ERO_F  = df


#-------------------
run = "JSN_01"
predictions_path = runs_to_examine[run]["predictions_raw__test"]
src = predictions_path

df = combine_predictions([src])
df["patientId_date_HF_LR"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:4]))
df["patientId_date_HF"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:3]))
df["patientId_date"] = df["file_name"].apply(lambda x: "_".join(x.split("_")[:2]))


# # For now correct predictions... Class 6 can not occur anyhow...
# df["labels"] = df["labels"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment="over_limit_to_NA"))
# df["preds"] = df["preds"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment="over_limit_to_NA"))
# df = df.dropna()
df_JSN  = df






In [ ]:
#### INPUT:
FRACTION_REQUIRED_VALID_SCORES = 0.75



# Model vs Ground truth

#### ERO H

In [ ]:
df = df_ERO_H
df_summed_H = sum_and_extrapolate_scores_df_ERO_H(df, fraction_required_valid_scores=FRACTION_REQUIRED_VALID_SCORES, limit_treatment_ED="E_D_mean")
df_delta_H = generate_score_differences(df_summed_H)

plot_SHS_sums(df_summed_H, figsize=(5,5))
plt.show()

# plot_SHS_deltas(df_delta_H)
# plt.show()

### ERO F: 


In [ ]:


df = df_ERO_F
df_summed_F = sum_and_extrapolate_scores_df_ERO_F(df, fraction_required_valid_scores=FRACTION_REQUIRED_VALID_SCORES)


df_delta_F = generate_score_differences(df_summed_F)
plot_SHS_sums(df_summed_F, figsize=(5,5))
plt.show()

df.head(1)

### ERO F + H: 

In [ ]:
# Plot for ERO model vs Gabi
df_summed_ERO_H_F = (
    df_summed_H
    .set_index('patientId_date')
    .add(df_summed_F.set_index('patientId_date'), fill_value=None)
    .reset_index()
)


In [ ]:
plot_SHS_sums(df_summed_ERO_H_F.dropna(), name="ERO H+F", figsize=(5,5), plot_histograms=False, regplot=True)
plt.show()

#### JSN H

In [ ]:
# Plot for JSN H model vs Gabi
df = df_JSN
df_summed_JSN_H = sum_and_extrapolate_scores_df_JSN_H(df, fraction_required_valid_scores=FRACTION_REQUIRED_VALID_SCORES)
df_delta_JSN_H = generate_score_differences(df_summed_JSN_H)

plot_SHS_sums(df_summed_JSN_H, figsize=(5,5), name="JSN H", plot_histograms=False)
plt.show()

# plot_SHS_deltas(df_delta_JSN_H)
# plt.show()

#### JSN F

In [ ]:
# Plot for JSN F model vs Gabi
df = df_JSN
df_summed_JSN_F = sum_and_extrapolate_scores_df_JSN_F(df, fraction_required_valid_scores=FRACTION_REQUIRED_VALID_SCORES)
df_delta_JSN_F = generate_score_differences(df_summed_JSN_F)

plot_SHS_sums(df_summed_JSN_F, figsize=(5,5), name="JSN F")
plt.show()

# plot_SHS_deltas(df_delta_JSN_F)
# plt.show()

#### JSN H+F

In [ ]:
# Plot for JSN model vs Gabi
df_summed_H_F_JSN = (
    df_summed_JSN_H
    .set_index('patientId_date')
    .add(df_summed_JSN_F.set_index('patientId_date'), fill_value=None)
    .reset_index()
)

plot_SHS_sums(df_summed_H_F_JSN.dropna(), figsize=(5,5), name="JSN H+F", plot_histograms=False, regplot=True)
plt.show()

#### SvH

In [ ]:
# Plot for SvH model vs Gabi
df_summed_SvH = (
    df_summed_H_F_JSN
    .set_index('patientId_date')
    .add(df_summed_ERO_H_F.set_index('patientId_date'), fill_value=None)
    .reset_index()
)

plot_SHS_sums(df_summed_SvH.dropna(), figsize=(5,5), name="SvH H+F", plot_histograms=False, regplot=True)
plt.show()

df_delta_SvH = generate_score_differences(df_summed_SvH)
plot_SHS_deltas(df_delta_SvH.dropna(), figsize=(5,5), name="SvH H+F", regplot=True)
plt.show()


# Double reading scores: 

In [ ]:
# Read double scoring data: 
double_score_path = '/home/cwatzenboeck/data/AutoPIX_cirdata/projects__autoscora/autoscoRA_TDEIMEL_HOME/autoscoRA_Preprocessing/output/double_scoring/Re_Scoring_50_v2_FINAL_pseudonymized_colrenamed.csv'
df_D = pd.read_csv(double_score_path)

# exclude those where we dont have a patient ID
m = ~df_D["id_nr"].isna()
df_D_excluded = df_D[~m]
df_D = df_D[m].copy()
df_D["id_nr"] = df_D["id_nr"].astype(int)
ids_double_scored = list(set(df_D["id_nr"]))

df_D["img_id_merge"] = df_D.apply(double_scoring_make_merge_id, axis=1)
df_D["patientId_date_HF_LR"] = df_D["img_id_merge"]    
df_D["patientId_visitDate"] = df_D["img_id_merge"].apply(lambda x: "_".join(x.split("_")[:2]))
df_D["patientId_date"] = df_D["patientId_visitDate"]


with resources.files("ra_utils.resources.scores_metadata").joinpath("roi_scores_matching.csv") as f:
    df_scores_meta = pd.read_csv(f)
H_ERO_scores = sorted(df_scores_meta[(df_scores_meta["ERO_or_JSN"] == "ERO") & (df_scores_meta["region"] == "H")]["score_name"].unique())
F_ERO_scores = sorted(df_scores_meta[(df_scores_meta["ERO_or_JSN"] == "ERO") & (df_scores_meta["region"] == "F")]["score_name"].unique())
H_JSN_scores = sorted(df_scores_meta[(df_scores_meta["ERO_or_JSN"] == "JSN") & (df_scores_meta["region"] == "H")]["score_name"].unique())
F_JSN_scores = sorted(df_scores_meta[(df_scores_meta["ERO_or_JSN"] == "JSN") & (df_scores_meta["region"] == "F")]["score_name"].unique())


df_D_H = df_D[df_D["bodypart_manual"] == "H"]
df_D_F = df_D[df_D["bodypart_manual"] == "F"]




df = df_D_H[H_ERO_scores + ["patientId_date_HF_LR", "patientId_date"]].copy()
df["JSN_or_ERO"] = "ERO"
df_long_H_ERO = df.melt(
    id_vars=["patientId_date_HF_LR", "patientId_date", "JSN_or_ERO"],
    value_vars=H_ERO_scores,
    var_name="score_type",
    value_name="scores"
)


# For F‐ERO
df = df_D_F[F_ERO_scores + ["patientId_date_HF_LR", "patientId_date"]].copy()
df["JSN_or_ERO"] = "ERO"
df_long_F_ERO = df.melt(
    id_vars=["patientId_date_HF_LR", "patientId_date", "JSN_or_ERO"],
    value_vars=F_ERO_scores,
    var_name="score_type",
    value_name="scores"
)

# For H‐JSN
df = df_D_H[H_JSN_scores + ["patientId_date_HF_LR", "patientId_date"]].copy()
df["JSN_or_ERO"] = "JSN"
df_long_H_JSN = df.melt(
    id_vars=["patientId_date_HF_LR", "patientId_date", "JSN_or_ERO"],
    value_vars=H_JSN_scores,
    var_name="score_type",
    value_name="scores"
)

# For F‐JSN
df = df_D_F[F_JSN_scores + ["patientId_date_HF_LR", "patientId_date"]].copy()
df["JSN_or_ERO"] = "JSN"
df_long_F_JSN = df.melt(
    id_vars=["patientId_date_HF_LR", "patientId_date", "JSN_or_ERO"],
    value_vars=F_JSN_scores,
    var_name="score_type",
    value_name="scores"
)


LIMIT_TREATMENT_DS =  "over_limit_to_NA" ## "over_limit_to_NA"  "over_limit_to_limit_plus_1"  over_limit_to_limit
df_long_H_ERO["scores"] = df_long_H_ERO["scores"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment=LIMIT_TREATMENT_DS))
df_long_F_ERO["scores"] = df_long_F_ERO["scores"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment=LIMIT_TREATMENT_DS))
df_long_H_JSN["scores"] = df_long_H_JSN["scores"].apply(lambda x: limit_treatment_number(x, limit=4, limit_treatment=LIMIT_TREATMENT_DS))
df_long_F_JSN["scores"] = df_long_F_JSN["scores"].apply(lambda x: limit_treatment_number(x, limit=4, limit_treatment=LIMIT_TREATMENT_DS))


df_all_long_D = pd.concat([
    df_long_H_ERO,
    df_long_F_ERO,
    df_long_H_JSN,
    df_long_F_JSN
], ignore_index=True)


In [ ]:
### Read scores origin: 
config = ra_utils.utils.config_parser.load_config(
    default_config="/home/cwatzenboeck/code/RA/ra_utils/runs/config_scoring/Exp06/dev.yml", 
    debugging_in_jupyter_nb=True, silencium=True)

src = config["data"]["score_path_H"]
df = pd.read_csv(src)
df["img_id_merge"] = df["img_id"].apply(lambda x: "_".join(x.split("_")[:4]))
df["patientId_visitDate"] = df["img_id_merge"].apply(lambda x: "_".join(x.split("_")[:2]))
df["bodypart_manual"] = "H" #!!!
df_H = df


src = config["data"]["score_path_F"]
df = pd.read_csv(src)
df["img_id_merge"] = df["img_id"].apply(lambda x: "_".join(x.split("_")[:4]))
df["patientId_visitDate"] = df["img_id_merge"].apply(lambda x: "_".join(x.split("_")[:2]))
df["bodypart_manual"] = "F" #!!!
df_F = df

scores_H =  [c for c in df_H.columns if c.startswith("r_")]
scores_F =  [c for c in df_F.columns if c.startswith("r_")]
assert (set(scores_H) & set(scores_F))  == set(), "Scores seem to be named same for hands and feet! Keep sperate!"

df_reader1 = pd.concat([df_F, df_H])

df_reader1["patientId_date_HF_LR"] = df_reader1["img_id"].apply(lambda x: "_".join(x.split("_")[:4]))
df_reader1["patientId_date"] = df_reader1["patientId_date_HF_LR"].apply(lambda x: "_".join(x.split("_")[:2]))

score_types = [s[2:] for s in df_reader1.columns if s.startswith("r_")]
rename_score_types_dct = {f"r_{k}": f"{k}" for k in score_types}
df_reader1 = df_reader1.rename(columns = rename_score_types_dct)

df_reader1_H = df_reader1[df_reader1["bodypart_manual"] == "H"]
df_reader1_F = df_reader1[df_reader1["bodypart_manual"] == "F"]


df_reader1_H[H_ERO_scores + ["patientId_date_HF_LR", "patientId_date"]].head(1)

df = df_reader1_H[H_ERO_scores + ["patientId_date_HF_LR", "patientId_date"]].copy()
df["JSN_or_ERO"] = "ERO"
df_long_H_ERO = df.melt(
    id_vars=["patientId_date_HF_LR", "patientId_date", "JSN_or_ERO"],
    value_vars=H_ERO_scores,
    var_name="score_type",
    value_name="scores"
)


# For F‐ERO
df = df_reader1_F[F_ERO_scores + ["patientId_date_HF_LR", "patientId_date"]].copy()
df["JSN_or_ERO"] = "ERO"
df_long_F_ERO = df.melt(
    id_vars=["patientId_date_HF_LR", "patientId_date", "JSN_or_ERO"],
    value_vars=F_ERO_scores,
    var_name="score_type",
    value_name="scores"
)

# For H‐JSN
df = df_reader1_H[H_JSN_scores + ["patientId_date_HF_LR", "patientId_date"]].copy()
df["JSN_or_ERO"] = "JSN"
df_long_H_JSN = df.melt(
    id_vars=["patientId_date_HF_LR", "patientId_date", "JSN_or_ERO"],
    value_vars=H_JSN_scores,
    var_name="score_type",
    value_name="scores"
)

# For F‐JSN
df = df_reader1_F[F_JSN_scores + ["patientId_date_HF_LR", "patientId_date"]].copy()
df["JSN_or_ERO"] = "JSN"
df_long_F_JSN = df.melt(
    id_vars=["patientId_date_HF_LR", "patientId_date", "JSN_or_ERO"],
    value_vars=F_JSN_scores,
    var_name="score_type",
    value_name="scores"
)

df_long_H_ERO["scores"] = df_long_H_ERO["scores"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment=LIMIT_TREATMENT_DS))
df_long_F_ERO["scores"] = df_long_F_ERO["scores"].apply(lambda x: limit_treatment_number(x, limit=5, limit_treatment=LIMIT_TREATMENT_DS))
df_long_H_JSN["scores"] = df_long_H_JSN["scores"].apply(lambda x: limit_treatment_number(x, limit=4, limit_treatment=LIMIT_TREATMENT_DS))
df_long_F_JSN["scores"] = df_long_F_JSN["scores"].apply(lambda x: limit_treatment_number(x, limit=4, limit_treatment=LIMIT_TREATMENT_DS))




df_all_long_reader1 = pd.concat([
    df_long_H_ERO,
    df_long_F_ERO,
    df_long_H_JSN,
    df_long_F_JSN
], ignore_index=True)



df = df_all_long_reader1
df["merge_id"] = df["patientId_date_HF_LR"] + "_" + df["JSN_or_ERO"] + "_" +  df["score_type"]
df1 = df

df = df_all_long_D
df["merge_id"] = df["patientId_date_HF_LR"] + "_" + df["JSN_or_ERO"] + "_" +  df["score_type"]
df2 = df

dfm = pd.merge(df2.rename(columns={"scores": "preds"}), df1[["merge_id", "scores"]].rename(columns={"scores": "labels"}), on="merge_id", how="left")



In [ ]:
# JSN F
df = dfm.dropna()
df_summed_JSN_F__D = sum_and_extrapolate_scores_df_JSN_F(df, fraction_required_valid_scores=FRACTION_REQUIRED_VALID_SCORES)

plot_SHS_sums(df_summed_JSN_F__D, figsize=(5,5), name="JSN F")
plt.show()

# plot_SHS_deltas(df_delta_JSN_F)
# plt.show()

In [ ]:
# JSN H
df = dfm
df_summed_JSN_H__D = sum_and_extrapolate_scores_df_JSN_H(df, fraction_required_valid_scores=FRACTION_REQUIRED_VALID_SCORES)
plot_SHS_sums(df_summed_JSN_H__D, figsize=(5,5), name="JSN H", plot_histograms=False, regplot=True)
plt.show()

# plot_SHS_deltas(df_delta_JSN_F)
# plt.show()

### SUM JSN

In [ ]:
# JSN H + F
df_summed_H_F_JSN__D = (
    df_summed_JSN_H__D
    .set_index('patientId_date')
    .add(df_summed_JSN_F__D.set_index('patientId_date'), fill_value=None)
    .reset_index()
)
fig, (ax_histx, ax_scatter, ax_histy) = plot_SHS_sums(df_summed_H_F_JSN__D.dropna(), figsize=(4,4), 
                                                      name="JSN H+F", plot_histograms=False,  regplot=True) #(0, 120))
ax_scatter.set_xlim(0,100)
ax_scatter.set_ylim(0,100)
ax_scatter.set_xticks(np.arange(0,150,30))
ax_scatter.set_yticks(np.arange(0,150,30))
plt.show()

### ERO F, H, H+F

In [ ]:
#### ERO F
df = dfm
df_summed_ERO_F__D = sum_and_extrapolate_scores_df_ERO_F(df, fraction_required_valid_scores=FRACTION_REQUIRED_VALID_SCORES)
plot_SHS_sums(df_summed_ERO_F__D, figsize=(5,5), name="ERO F", plot_histograms=False, regplot=False)
plt.show()

In [ ]:
### ERO H
df = dfm
df_summed_ERO_H__D = sum_and_extrapolate_scores_df_ERO_H(df, fraction_required_valid_scores=FRACTION_REQUIRED_VALID_SCORES)
plot_SHS_sums(df_summed_ERO_H__D, figsize=(5,5), name="ERO H", plot_histograms=False, regplot=True)
plt.show()

### SUM ERO: 

In [ ]:
# ERO H+F
df_summed_H_F_ERO__D = (
    df_summed_ERO_H__D
    .set_index('patientId_date')
    .add(df_summed_ERO_F__D.set_index('patientId_date'), fill_value=None)
    .reset_index()
)
fig, (ax_histx, ax_scatter, ax_histy) = plot_SHS_sums(df_summed_H_F_ERO__D.dropna(), figsize=(4,4), name="ERO H+F", plot_histograms=False, regplot=True) #(0, 120))
# ax_scatter.set_xlim(0,100)
# ax_scatter.set_ylim(0,100)
# ax_scatter.set_xticks(np.arange(0,150,30))
# ax_scatter.set_yticks(np.arange(0,150,30))
plt.show()

### SUM SvH

In [ ]:
# SvH
df_summed_H_F_SvH__D = (
    df_summed_H_F_ERO__D
    .set_index('patientId_date')
    .add(df_summed_H_F_JSN__D.set_index('patientId_date'), fill_value=None)
    .reset_index()
)
fig, (ax_histx, ax_scatter, ax_histy) = plot_SHS_sums(df_summed_H_F_SvH__D.dropna(), figsize=(4,4), name="H+F SvH", plot_histograms=False, regplot=True) #(0, 120))
ax_scatter.set_xlim(0,200)
ax_scatter.set_ylim(0,200)
ax_scatter.set_xticks(np.arange(0,200,30))
ax_scatter.set_yticks(np.arange(0,200,30))
plt.show()